In [1]:
from os import chdir
from pathlib import Path

cwd = Path.cwd()
print(f"CWD: {cwd}")
if cwd.name == "code":
    chdir("..")
print(f"CWD: {Path.cwd()}")

CWD: /home/james/git/research/fed-learning/code
CWD: /home/james/git/research/fed-learning


In [2]:
import pandas as pd

from torch.optim import SGD
import sys

In [3]:
from src.models.MatrixFactorization import MF, UMF
from src.graphs import random_k_out_graph, create_random_5_out_graph
from src.users import User
from src.training.decentralized import decentralized_train_loop, decentralized_validate_loop
from src.data_utils import create_dataloader

In [4]:
def get_size(obj, seen=None):
    """Recursively finds size of objects"""
    size = sys.getsizeof(obj)
    if seen is None:
        seen = set()
    obj_id = id(obj)
    if obj_id in seen:
        return 0
    # Important mark as seen *before* entering recursion to gracefully handle
    # self-referential objects
    seen.add(obj_id)
    if isinstance(obj, dict):
        size += sum([get_size(v, seen) for v in obj.values()])
        size += sum([get_size(k, seen) for k in obj.keys()])
    elif hasattr(obj, '__dict__'):
        size += get_size(obj.__dict__, seen)
    elif hasattr(obj, '__iter__') and not isinstance(obj, (str, bytes, bytearray)):
        size += sum([get_size(i, seen) for i in obj])
    return size

In [5]:
train_df = pd.read_csv("dataset/hm_train_df.csv")
val_df = pd.read_csv("dataset/hm_val_df.csv")
n_users = train_df['customer_id'].nunique()
n_items = train_df['product_code'].nunique()

In [6]:
train_dl = create_dataloader(train_df, "urs")
graph = create_random_5_out_graph(n_users)

In [7]:
lr=.01
weight_decay = 0.01
mom = 0.01
n_factors = 30
sparse = False

In [8]:
mf_users = {}
for i in range(n_users):
    model = MF(n_users, n_items, n_factors=n_factors, sparse=sparse)
    mf_users[i] = User(
        id=i,
        model=model,
        optimizer=SGD(model.parameters(), lr=lr, weight_decay=weight_decay, momentum=mom),
    )

In [9]:
get_size(mf_users)

29191317

In [10]:
decentralized_train_loop(mf_users, train_dl, graph)

  0%|          | 0/1760 [00:00<?, ?it/s]

34.40803407293037

In [11]:
get_size(mf_users)

32415701

In [8]:
umf_users = {}
for i in range(n_users):
    model = UMF(n_items, n_factors=n_factors, sparse=sparse)
    umf_users[i] = User(
        id=i,
        model=model,
        optimizer=SGD(model.parameters(), lr=lr, weight_decay=weight_decay, momentum=mom),
    )

In [9]:
get_size(umf_users)

29191289

In [10]:
decentralized_train_loop(umf_users, train_dl, graph)

  0%|          | 0/1760 [00:00<?, ?it/s]

34.561236586060076

In [11]:
get_size(umf_users)

32415673